# Machine Learning Lab — Module 2
## Feature Engineering & Linear Regression

### Lab focus
1. Manual categorical encoding
2. Principal Component Analysis (PCA) using covariance matrix and eigen decomposition
3. Simple linear regression using the least-squares method with NumPy

**Learning goal:** Convert raw data into model-ready numerical features, reduce dimensionality with PCA, and implement simple linear regression from the mathematical least-squares solution.

## 1. Required Libraries

We will use:

- **NumPy** — numerical computation
- **Pandas** — convenient tabular data handling
- **Matplotlib** — visualization

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Part A — Manual Encoding

## 2. Why Encoding?

Machine-learning algorithms generally operate on numerical representations.

Suppose a feature contains:

`red`, `green`, `blue`

A simple manual encoding can map:

- red → 0
- green → 1
- blue → 2

This is called **label/manual encoding**.

> Note: assigning integers introduces an artificial ordering. For genuinely nominal categories, one-hot encoding is often preferable in practical ML workflows.

In [8]:
colors = ["red", "green", "blue", "green", "red", "blue"]

encoding = {
    "red": 0,
    "green": 1,
    "blue": 2
}

encoded_colors = [encoding[color] for color in colors]

print("Original :", colors)
print("Encoded  :", encoded_colors)

Original : ['red', 'green', 'blue', 'green', 'red', 'blue']
Encoded  : [0, 1, 2, 1, 0, 2]


## 3. Encoding a DataFrame Column

Let's apply the same idea to a small dataset.

In [ ]:
df = pd.DataFrame({
    "Student": ["S1", "S2", "S3", "S4", "S5"],
    "Color": ["red", "green", "blue", "green", "red"],
    "StudyHours": [2, 4, 6, 5, 3]
})

print(df)

In [ ]:
color_map = {"red": 0, "green": 1, "blue": 2}

df["Color_Encoded"] = df["Color"].map(color_map)

print(df)

### Practice

Create your own dictionary to encode three categories of your choice.

Example:

`Low → 0, Medium → 1, High → 2`

In [ ]:
levels = ["Low", "High", "Medium", "High", "Low"]

level_map = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

encoded_levels = [level_map[x] for x in levels]
print(encoded_levels)

# Part B — Principal Component Analysis (PCA)

## 4. PCA Idea

PCA transforms correlated features into a smaller set of new features called **principal components**.

For this lab, we follow the mathematical pipeline:

1. Arrange the numerical data matrix
2. Center each feature
3. Compute the covariance matrix
4. Compute eigenvalues and eigenvectors
5. Select the principal component(s) with the largest eigenvalues
6. Project the data onto the selected component(s)

The eigenvector with the largest eigenvalue gives the direction of maximum variance.

## 5. Create a Small 2-D Dataset

We use two numerical features with a visible relationship.

In [ ]:
X = np.array([
    [2.0, 1.0],
    [3.0, 2.0],
    [4.0, 3.0],
    [5.0, 4.0],
    [6.0, 5.0]
])

print(X)

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], X[:, 1])
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Original 2-D Data")
plt.grid(True)
plt.show()

## 6. Step 1 — Center the Data

PCA is commonly applied after subtracting the mean of each feature.

\[
X_c = X - \mu
\]

where \(\mu\) is the vector of feature means.

In [ ]:
mean = np.mean(X, axis=0)
X_centered = X - mean

print("Feature means:", mean)
print("\nCentered data:\n", X_centered)

## 7. Step 2 — Covariance Matrix

The sample covariance matrix is:

\[
C = \frac{1}{n-1}X_c^T X_c
\]

It describes how the features vary together.

In [ ]:
cov_matrix = np.cov(X_centered, rowvar=False)

print("Covariance matrix:\n", cov_matrix)

## 8. Step 3 — Eigen Decomposition

Find:

- eigenvalues → amount of variance represented by each component
- eigenvectors → directions of the principal components

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

print("Eigenvalues:", eigenvalues)
print("\nEigenvectors (columns):\n", eigenvectors)

## 9. Step 4 — Sort Principal Components

We want the component with the **largest eigenvalue** first.

In [ ]:
order = np.argsort(eigenvalues)[::-1]

eigenvalues_sorted = eigenvalues[order]
eigenvectors_sorted = eigenvectors[:, order]

print("Sorted eigenvalues:", eigenvalues_sorted)
print("\nSorted eigenvectors:\n", eigenvectors_sorted)

## 10. Explained Variance Ratio

A useful interpretation is:

\[
EVR_i = \frac{\lambda_i}{\sum_j \lambda_j}
\]

This tells us the fraction of total variance explained by each principal component.

In [ ]:
explained_variance_ratio = eigenvalues_sorted / np.sum(eigenvalues_sorted)

print("Explained variance ratio:", explained_variance_ratio)
print("Total:", np.sum(explained_variance_ratio))

## 11. Step 5 — Project onto the First Principal Component

Select the first eigenvector and project the centered data:

\[
Z = X_c W
\]

where \(W\) contains the selected eigenvector(s).

In [ ]:
W = eigenvectors_sorted[:, :1]   # keep one principal component
Z = X_centered @ W

print("First principal component direction:\n", W)
print("\nReduced 1-D representation:\n", Z)

## 12. PCA from Scratch — Complete Function

The following function packages the steps into one reusable implementation.

In [ ]:
def pca_from_scratch(X, n_components):
    X = np.asarray(X, dtype=float)

    # 1. Center
    mean = np.mean(X, axis=0)
    X_centered = X - mean

    # 2. Covariance matrix
    covariance = np.cov(X_centered, rowvar=False)

    # 3. Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eig(covariance)

    # 4. Sort by decreasing eigenvalue
    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    # 5. Select components
    components = eigenvectors[:, :n_components]

    # 6. Project
    X_reduced = X_centered @ components

    explained_ratio = eigenvalues / np.sum(eigenvalues)

    return X_reduced, components, eigenvalues, explained_ratio, mean


X_reduced, components, values, ratios, mean = pca_from_scratch(X, 1)

print("Reduced data:\n", X_reduced)
print("\nComponents:\n", components)
print("\nEigenvalues:", values)
print("\nExplained variance ratio:", ratios)

### PCA Practice Problem

Use the dataset:

```text
X = [[2, 1],
     [3, 2],
     [5, 4],
     [6, 5],
     [8, 7]]
```

Tasks:

1. Center the data.
2. Calculate the covariance matrix.
3. Find eigenvalues and eigenvectors.
4. Identify the first principal component.
5. Calculate the explained variance ratio.
6. Reduce the data from 2 dimensions to 1 dimension.

In [ ]:
X_practice = np.array([
    [2, 1],
    [3, 2],
    [5, 4],
    [6, 5],
    [8, 7]
], dtype=float)

# Try solving manually with NumPy functions.
mean = np.mean(X_practice, axis=0)
Xc = X_practice - mean
cov = np.cov(Xc, rowvar=False)
vals, vecs = np.linalg.eig(cov)
order = np.argsort(vals)[::-1]
vals = vals[order]
vecs = vecs[:, order]

print("Mean:", mean)
print("\nCovariance:\n", cov)
print("\nEigenvalues:", vals)
print("\nEigenvectors:\n", vecs)
print("\nExplained variance ratio:", vals / vals.sum())

Z = Xc @ vecs[:, :1]
print("\n1-D PCA representation:\n", Z)

# Part C — Simple Linear Regression

## 13. Problem Setup

Simple linear regression models the relationship between one input \(x\) and an output \(y\):

\[
\hat{y} = b_0 + b_1x
\]

where:

- \(b_0\) = intercept
- \(b_1\) = slope
- \(\hat y\) = predicted value

In this lab we use the **least-squares** solution rather than gradient descent.

In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([3, 5, 7, 9, 11], dtype=float)

print("x:", x)
print("y:", y)

## 14. Least-Squares Formula

For simple linear regression:

\[
b_1 =
\frac{\sum (x_i-\bar{x})(y_i-\bar{y})}
{\sum (x_i-\bar{x})^2}
\]

and

\[
b_0 = \bar{y} - b_1\bar{x}
\]

In [ ]:
x_mean = np.mean(x)
y_mean = np.mean(y)

b1 = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
b0 = y_mean - b1 * x_mean

print("Slope (b1):", b1)
print("Intercept (b0):", b0)

## 15. Make Predictions

Now use:

\[
\hat y = b_0 + b_1x
\]

In [ ]:
y_pred = b0 + b1 * x

print("Actual values:   ", y)
print("Predicted values:", y_pred)

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(x, y, label="Actual data")
plt.plot(x, y_pred, label="Regression line")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Simple Linear Regression")
plt.legend()
plt.grid(True)
plt.show()

## 16. Least Squares Using a Matrix Equation

The same idea can be written as:

\[
\beta = (X^T X)^{-1}X^T y
\]

where:

\[
X =
\begin{bmatrix}
1 & x_1 \\
1 & x_2 \\
\vdots & \vdots \\
1 & x_n
\end{bmatrix}
\]

The first column of ones represents the intercept.

In [ ]:
X_design = np.column_stack((np.ones(len(x)), x))

beta = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y

print("Design matrix:\n", X_design)
print("\nBeta [intercept, slope]:", beta)

y_pred_matrix = X_design @ beta
print("\nPredictions:", y_pred_matrix)

### Better Numerical Practice

For real numerical work, `np.linalg.solve()` is generally preferable to explicitly calculating a matrix inverse when solving linear systems.

For this teaching lab, the inverse form is shown because it directly matches the normal-equation formula.

In [ ]:
A = X_design.T @ X_design
b = X_design.T @ y

beta_solve = np.linalg.solve(A, b)

print("Beta using np.linalg.solve:", beta_solve)

## 17. Mini Practice Problem — Study Hours vs Marks

Suppose:

| Study Hours | Marks |
|---:|---:|
| 1 | 35 |
| 2 | 42 |
| 3 | 50 |
| 4 | 58 |
| 5 | 65 |

Tasks:

1. Calculate the slope and intercept using the least-squares formulas.
2. Build the regression equation.
3. Predict marks for 6 study hours.
4. Plot the observations and regression line.

In [ ]:
study_hours = np.array([1, 2, 3, 4, 5], dtype=float)
marks = np.array([35, 42, 50, 58, 65], dtype=float)

x_mean = study_hours.mean()
y_mean = marks.mean()

slope = np.sum((study_hours - x_mean) * (marks - y_mean)) / np.sum((study_hours - x_mean) ** 2)
intercept = y_mean - slope * x_mean

prediction_6 = intercept + slope * 6

print("Slope:", slope)
print("Intercept:", intercept)
print(f"Regression equation: y = {intercept:.3f} + {slope:.3f}x")
print("Predicted marks for 6 hours:", prediction_6)

line = intercept + slope * study_hours

plt.figure(figsize=(6, 4))
plt.scatter(study_hours, marks, label="Observed marks")
plt.plot(study_hours, line, label="Least-squares line")
plt.xlabel("Study Hours")
plt.ylabel("Marks")
plt.title("Study Hours vs Marks")
plt.legend()
plt.grid(True)
plt.show()

## 18. Module 2 Summary

### Feature Engineering
- Convert categorical values into numerical representations.
- Manual/label encoding is simple, but numeric codes can create an artificial ordering.

### PCA
\[
X \rightarrow X_c \rightarrow Cov(X) \rightarrow
\text{Eigenvalues/Eigenvectors} \rightarrow
\text{Principal Components} \rightarrow Z
\]

The eigenvector corresponding to the largest eigenvalue represents the direction of maximum variance.

### Linear Regression
\[
\hat y = b_0+b_1x
\]

For the least-squares solution:

\[
\beta=(X^TX)^{-1}X^Ty
\]

The same matrix/eigen concepts from Module 1 are reused here, making Module 2 a direct application of linear algebra to ML.

## 19. Final Lab Challenge

Build a small workflow that:

1. Creates a dataset with one categorical feature and two numerical features.
2. Manually encodes the categorical feature.
3. Applies PCA to the two numerical features.
4. Uses simple linear regression to predict an output from one numerical feature.
5. Displays the intermediate results clearly.

**Suggested extension:** Compare your PCA implementation with `sklearn.decomposition.PCA` after you have completed the NumPy implementation.